# 03 — The transformer block and the full GPT

Notebook 02 gave us multi-head attention: a way for tokens to communicate. That alone isn't a
language model. Here we assemble the rest and end with the complete `GPT` class — the same one in
[`model.py`](../model.py):

- the **feed-forward MLP** (where per-token computation happens),
- **residual connections** + **pre-norm LayerNorm** (what makes deep stacks trainable),
- the full **Block** = attention + MLP wrapped in those,
- **token + positional embeddings** (attention has no built-in sense of order),
- the **LM head** with **weight tying**,
- and the forward pass that turns token ids into a loss.

We'll motivate every choice — and name the alternative we rejected and why.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
torch.manual_seed(1337)

## 1. The MLP: "attention communicates, the MLP computes"

Attention *moves* information between positions but does little nonlinear processing — its output
is a *weighted average of values*, which is a fairly linear operation on v. The per-token
"thinking" happens in a position-wise feed-forward network: the same 2-layer MLP applied
independently to every position.

Shape journey: `C → 4C → C`. Expand to 4× width, apply a nonlinearity, project back.

**Why expand 4×?** The wide hidden layer is where the block's capacity lives — it gives the
nonlinearity room to represent many features before compressing back to the residual width. 4× is
the convention from the original transformer; it's empirical, not derived. (Later models trade
this: SwiGLU uses ~2.7× but with a gating branch — notebook 07.)

**Why GELU, not ReLU?** ReLU zeros every negative input, so a unit that's slightly negative gets
*exactly zero* gradient and can "die." GELU (`x·Φ(x)`, Φ = Gaussian CDF) is a smooth gate: small
negatives pass a little signal and keep a nonzero gradient. GPT-2/BERT validated it; it's the
default for this family.

In [2]:
class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.0, bias=False):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4 * n_embd, bias=bias)   # expand
        self.gelu   = nn.GELU()                                  # smooth nonlinearity
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=bias)   # project back
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

m = MLP(32)
print("MLP:  (B,T,32) ->", tuple(m(torch.randn(4, 8, 32)).shape), " (width preserved)")

# GELU vs ReLU near zero -- note GELU's smoothness and nonzero negative branch
xs = torch.tensor([-2., -0.5, 0., 0.5, 2.])
print("\nx    :", xs.tolist())
print("ReLU :", F.relu(xs).tolist())
print("GELU :", [round(v,3) for v in F.gelu(xs).tolist()])

MLP:  (B,T,32) -> (4, 8, 32)  (width preserved)

x    : [-2.0, -0.5, 0.0, 0.5, 2.0]
ReLU : [0.0, 0.0, 0.0, 0.5, 2.0]
GELU : [-0.046, -0.154, 0.0, 0.346, 1.954]


## 2. Residual connections + LayerNorm: why deep stacks train at all

Stack a dozen of these sublayers naively and the network won't train — gradients vanish or
explode on the way down. Two ingredients fix this.

**Residual connections.** Instead of `x = sublayer(x)`, we write `x = x + sublayer(x)`. Now there
is an **identity path** the gradient can flow through untouched: `d(out)/d(x)` always contains a
`+1` term, so even the earliest layer receives a clean gradient signal regardless of depth. Think
of the residual stream as a shared "notepad" that each sublayer *reads from and adds a correction
to*, never overwrites.

**LayerNorm.** Normalizes each token's vector to zero mean / unit variance across its features
(then rescales with learned gain/bias). This keeps activations at a consistent scale so no
sublayer receives wildly-sized inputs. Unlike BatchNorm it's computed **per token**, independent
of other examples in the batch — essential for variable-length, autoregressive generation where
batch statistics aren't available.

**Pre-norm vs post-norm — the placement matters.** Two ways to combine them:

| layout | formula | behavior |
|---|---|---|
| **post-norm** (2017 paper) | `x = LayerNorm(x + sublayer(x))` | norm sits *on* the residual path; the clean identity route is repeatedly renormalized, gradients degrade with depth, needs careful LR warmup |
| **pre-norm** (GPT-2 onward) | `x = x + sublayer(LayerNorm(x))` | norm sits *inside* the branch; the residual stream is never touched, so the identity path stays pristine and very deep stacks train stably |

We use **pre-norm**. It's what makes 100+ layer models feasible and it's far more forgiving of
learning rate. Let's *demonstrate* the difference in gradient flow.

In [3]:
def grad_to_first_layer(prenorm, depth=40, C=64):
    '''Push one signal through `depth` residual sublayers, measure the gradient
       that reaches the very first layer. Bigger = healthier training signal.'''
    torch.manual_seed(0)
    layers = nn.ModuleList([nn.Sequential(nn.LayerNorm(C), nn.Linear(C, C)) for _ in range(depth)])
    x = torch.randn(1, 8, C, requires_grad=True)
    h = x
    for layer in layers:
        if prenorm:                       # x = x + f(LN(x))   -> LN inside branch
            h = h + layer(h)
        else:                             # x = LN(x + f(x))   -> LN on main path
            norm, lin = layer[0], layer[1]
            h = norm(h + lin(h))
    h.sum().backward()
    return x.grad.abs().mean().item()

pre  = grad_to_first_layer(prenorm=True)
post = grad_to_first_layer(prenorm=False)
print(f"gradient reaching layer 1 through a 40-deep stack:")
print(f"  pre-norm : {pre:.4e}")
print(f"  post-norm: {post:.4e}")
print(f"  pre-norm delivers ~{pre/post:.0f}x more gradient signal to the bottom layer")

gradient reaching layer 1 through a 40-deep stack:
  pre-norm : 2.5896e+00
  post-norm: 5.1604e-09
  pre-norm delivers ~501818830x more gradient signal to the bottom layer


## 3. One Block

A block is just: pre-norm attention with a residual, then pre-norm MLP with a residual.

```
x = x + attn(LayerNorm(x))
x = x + mlp (LayerNorm(x))
```

Two separate LayerNorms (one per sublayer), two separate residual adds. That's the entire
repeating unit of a transformer — everything else is stacking these and bolting on I/O.

In [4]:
# import the REAL modules from model.py so this notebook documents the shipping code
import sys; sys.path.insert(0, "..")
from model import CausalSelfAttention, Block, GPT, GPTConfig

cfg = GPTConfig(vocab_size=65, block_size=8, n_layer=1, n_head=4, n_embd=32, dropout=0.0)
block = Block(cfg)
x = torch.randn(4, 8, 32)
print("Block:  (B,T,C) ->", tuple(block(x).shape), " (shape preserved, so blocks stack)")
print("\nBlock structure:"); print(block)

Block:  (B,T,C) -> (4, 8, 32)  (shape preserved, so blocks stack)

Block structure:
Block(
  (ln_1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=False)
  (attn): CausalSelfAttention(
    (c_attn): Linear(in_features=32, out_features=96, bias=False)
    (c_proj): Linear(in_features=32, out_features=32, bias=False)
    (attn_dropout): Dropout(p=0.0, inplace=False)
    (resid_dropout): Dropout(p=0.0, inplace=False)
  )
  (ln_2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=False)
  (mlp): MLP(
    (c_fc): Linear(in_features=32, out_features=128, bias=False)
    (gelu): GELU(approximate='none')
    (c_proj): Linear(in_features=128, out_features=32, bias=False)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


Notice the output shape equals the input shape — `(B, T, C)` in, `(B, T, C)` out. That invariance
is *why* we can stack N blocks: each one reads the residual stream and writes an update back, and
the stream keeps the same shape all the way up.

## 4. Embeddings — and why we need *positional* ones

The model can't consume integer token ids directly; each id indexes a **token embedding** table
`wte` of shape `(vocab_size, C)`, turning ids into learnable vectors.

But there's a subtlety unique to attention: **it is permutation-equivariant.** Attention is a
weighted sum over positions — it has *no inherent notion of order*. Shuffle the input tokens and
(ignoring the mask) the outputs shuffle identically; "dog bites man" and "man bites dog" would be
indistinguishable. We must inject order explicitly.

We use **learned absolute positional embeddings**: a second table `wpe` of shape `(block_size,
C)`, one vector per position 0…T−1, **added** to the token embeddings. Simple, and exactly what
GPT-2 did. (Alternatives — sinusoidal from the original paper, or RoPE which rotates q/k by
position — are covered in notebook 07. RoPE is what modern models use because it generalizes to
longer contexts; learned absolute embeddings can't represent positions beyond `block_size`.)

Let's prove attention is order-blind without positions:

In [5]:
# Pure attention (mask removed) is PERMUTATION-EQUIVARIANT: shuffle the input
# tokens and the outputs come out shuffled the SAME way, with identical values.
# There is no way to read absolute position out of it.
full = CausalSelfAttention(GPTConfig(vocab_size=65, block_size=8, n_head=4, n_embd=32, dropout=0.0))
full.eval()
full.mask.fill_(1.0)                        # remove the causal mask to show pure attention
x = torch.randn(1, 8, 32)
perm = torch.randperm(8)
with torch.no_grad():
    out_then_perm = full(x)[:, perm, :]     # attend, then shuffle the outputs
    perm_then_out = full(x[:, perm, :])     # shuffle the inputs, then attend
print("attention(shuffle(x)) == shuffle(attention(x)) :",
      torch.allclose(out_then_perm, perm_then_out, atol=1e-5))
print("=> attention carries no notion of position; 'dog bites man' and 'man bites")
print("   dog' are the same bag of vectors to it. That's why we add wpe.")

attention(shuffle(x)) == shuffle(attention(x)) : True
=> attention carries no notion of position; 'dog bites man' and 'man bites
   dog' are the same bag of vectors to it. That's why we add wpe.


## 5. The LM head and weight tying

After the final block and a last LayerNorm, we project each token's C-dim vector to
`vocab_size` **logits** — a score for every possible next token. That's the `lm_head` linear
layer, shape `(C, vocab_size)`.

**Weight tying:** we set `lm_head.weight = wte.weight` — the output projection *reuses the input
embedding matrix*. Why it's principled: `wte` maps token id → vector; the head maps vector →
token scores. These are the two directions of the *same* correspondence between tokens and the
model's feature space, so sharing one matrix is natural. Practically it (a) removes
`vocab_size × C` parameters — for a big vocab that's a large fraction of the model — and (b) acts
as a regularizer that ties input and output representations, and it empirically improves loss.

In [6]:
model = GPT(GPTConfig(vocab_size=65, block_size=64, n_layer=4, n_head=4, n_embd=128, dropout=0.0))
print("lm_head and wte are the SAME tensor (tied):",
      model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr())

tied = model.wte.weight.numel()
print(f"weight tying saves {tied:,} params "
      f"({100*tied/sum(p.numel() for p in model.parameters()):.0f}% of this small model)")

lm_head and wte are the SAME tensor (tied): True
weight tying saves 8,320 params (1% of this small model)


## 6. The full GPT forward pass

Now the whole thing, end to end. Reading `GPT.forward`:

1. `tok_emb = wte(idx)` — look up token vectors, `(B, T, C)`.
2. `pos_emb = wpe(positions)` — look up position vectors, `(T, C)`, broadcast-added.
3. run through the stack of blocks (the residual stream evolving).
4. final LayerNorm, then `lm_head` → logits `(B, T, vocab_size)`.
5. if targets given, **cross-entropy** vs the next token at every position.

The training objective is beautifully simple: at each position, predict the **next** token.
`targets` is just `inputs` shifted left by one. One forward pass yields T next-token predictions
per sequence, so every token is a training signal.

In [7]:
# a tiny end-to-end pass on random "tokens"
B, T = 2, 16
idx     = torch.randint(0, 65, (B, T))
targets = torch.randint(0, 65, (B, T))
logits, loss = model(idx, targets)
print("logits:", tuple(logits.shape), " (a score for each of 65 tokens, at each position)")
print("loss  :", round(loss.item(), 3))

# sanity: an untrained model is ~uniform over the vocab, so loss ~ ln(vocab_size)
print("expected loss at init ~ ln(65) =", round(math.log(65), 3), " -> matches, model is 'guessing'")

logits: (2, 16, 65)  (a score for each of 65 tokens, at each position)
loss  : 4.211
expected loss at init ~ ln(65) = 4.174  -> matches, model is 'guessing'


In [8]:
# The target is the input shifted by one -- THIS is next-token prediction.
seq = torch.tensor([[8, 15, 23, 4, 42, 7]])      # some token ids
inp, tgt = seq[:, :-1], seq[:, 1:]
print("input :", inp.tolist()[0])
print("target:", tgt.tolist()[0], "  <- each target is the token that FOLLOWS the input at that position")
print("i.e. from [8] predict 15, from [8,15] predict 23, ...")

input : [8, 15, 23, 4, 42]
target: [15, 23, 4, 42, 7]   <- each target is the token that FOLLOWS the input at that position
i.e. from [8] predict 15, from [8,15] predict 23, ...


## 7. Parameter budget and initialization

Two details that matter for training well.

**Where the parameters live.** Counting them by component shows the MLP (the 4× expansion) and,
for small vocabs relative to width, the attention projections dominate — the embeddings are a
rounding error here but become huge when the vocab is 50k. This is why our TinyShakespeare models
(vocab 65) put nearly all capacity into the transformer, while a big-vocab model spends a lot on
`wte`.

In [9]:
def param_breakdown(model):
    groups = {}
    for name, p in model.named_parameters():
        key = ("token embedding (wte)" if "wte" in name else
               "position embedding (wpe)" if "wpe" in name else
               "attention" if "attn" in name else
               "MLP" if "mlp" in name else
               "layernorm" if "ln" in name else "other")
        groups[key] = groups.get(key, 0) + p.numel()
    total = sum(p.numel() for p in model.parameters())
    for k, v in sorted(groups.items(), key=lambda kv: -kv[1]):
        print(f"  {k:24s} {v:>10,d}  ({100*v/total:4.1f}%)")
    print(f"  {'TOTAL':24s} {total:>10,d}")

param_breakdown(model)

  MLP                         524,288  (65.2%)
  attention                   262,144  (32.6%)
  token embedding (wte)         8,320  ( 1.0%)
  position embedding (wpe)      8,192  ( 1.0%)
  layernorm                     1,152  ( 0.1%)
  TOTAL                       804,096


**Initialization.** Weights start as `N(0, 0.02²)` (GPT-2's choice). One extra trick in
`GPT._init_weights`: the **residual projections** (`c_proj` in both attention and MLP) are scaled
down by `1/√(2·n_layer)`. Reason: every block *adds* two contributions into the residual stream,
so without damping, the stream's variance would grow with depth and blow up in deep models.
Scaling the projections that write into the stream keeps its variance stable from bottom to top.

In [10]:
# show the residual-projection init is smaller than the standard 0.02
import numpy as np
stds = {}
for name, p in model.named_parameters():
    if name.endswith("c_proj.weight"):
        stds.setdefault("residual proj (c_proj)", []).append(p.std().item())
    elif name.endswith("c_fc.weight") or name.endswith("c_attn.weight"):
        stds.setdefault("normal linear", []).append(p.std().item())
for k, v in stds.items():
    print(f"{k:26s} mean std = {np.mean(v):.4f}")
print(f"\nexpected damped std = 0.02/sqrt(2*{model.config.n_layer}) = "
      f"{0.02/math.sqrt(2*model.config.n_layer):.4f}  -> matches the residual projections")

normal linear              mean std = 0.0200
residual proj (c_proj)     mean std = 0.0071

expected damped std = 0.02/sqrt(2*4) = 0.0071  -> matches the residual projections


## Takeaways

- A transformer block is **`x = x + attn(LN(x)); x = x + mlp(LN(x))`** — attention communicates
  between tokens, the MLP computes on each token; residuals + pre-norm keep the deep stack
  trainable (we measured pre-norm delivering ~orders of magnitude more gradient to the bottom
  layer than post-norm).
- **Residual stream** = a shared notepad every sublayer reads and adds to; the `+x` identity path
  is the highway gradients travel down.
- **LayerNorm** normalizes per token (not per batch), so it works during autoregressive
  generation. **Pre-norm** placement (inside the branch) is what modern GPTs use.
- Attention is **order-blind**, so we add **positional embeddings**; learned-absolute is GPT-2's
  choice (RoPE later).
- **Weight tying** shares `wte` with the LM head — fewer params, a useful regularizer, better loss.
- The objective is **next-token prediction** with cross-entropy; the target is the input shifted
  by one, giving T training signals per sequence. Loss at init ≈ ln(vocab_size) is your
  "is it wired up right?" check.
- The **residual-projection init scaling** (`1/√(2·n_layer)`) keeps the residual stream's
  variance stable with depth.

**Interview quick-fire:**
- *Why residual connections?* → they create an identity path so gradients reach early layers undiminished; without them deep transformers don't train.
- *Pre-norm vs post-norm?* → pre-norm keeps LayerNorm off the residual highway, giving stable gradients at depth and tolerance to high LR; post-norm (original paper) needs careful warmup.
- *Why does a transformer need positional embeddings?* → self-attention is permutation-equivariant; without position info it can't distinguish word orders.
- *What does weight tying do?* → reuses the embedding matrix as the output head — saves vocab×C params and regularizes.
- *What's the loss at initialization and why?* → ≈ ln(vocab_size), because an untrained model predicts ~uniformly over the vocabulary.

We now have a complete, correct GPT. **Next:** [04 — Training](04_training.ipynb): the loop that
turns this random-initialized model into one that writes Shakespeare — AdamW, LR schedule,
grad clip/accumulation, mixed precision, and the loss curves.